In [1]:
import pandas as pd
import numpy as np
import warnings

In [2]:
df = pd.read_parquet("../data/processed/ca1_long.parquet")
df.head()

,id,item_id,dept_id,cat_id,store_id,state_id,sales,d_num,date,wm_yr_wk,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap,sell_price
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,3,1,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0,2.0
1,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,0,2,2011-01-30,11101,2,1,2011,NaN,NaN,NaN,NaN,0,2.0
2,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,0,3,2011-01-31,11101,3,1,2011,NaN,NaN,NaN,NaN,0,2.0
3,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,1,4,2011-02-01,11101,4,2,2011,NaN,NaN,NaN,NaN,1,2.0
4,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,4,5,2011-02-02,11101,5,2,2011,NaN,NaN,NaN,NaN,1,2.0


In [3]:
# Cell 2: sort, clean
H = 28
LAST_DAY = int(df["d_num"].max())

first_day = (df.loc[df["sell_price"].notna()]
             .groupby("item_id", observed=True)["d_num"].min().rename("first_day"))
df = df.merge(first_day, on="item_id", how="left")
df = df[df["d_num"] >= df["first_day"]].copy()
df = df.sort_values(["item_id", "d_num"]).reset_index(drop=True)

for c in ["event_name_1", "event_type_1", "event_name_2", "event_type_2"]:
    df[c] = df[c].astype(object).fillna("none").astype("category")

df["age"] = (df["d_num"] - df["first_day"]).astype("int16")

g = df.groupby("item_id", observed=True)["d_num"].agg(["min", "max", "count"])
print("rows:", len(df))
print("every item consecutive:", ((g["max"] - g["min"] + 1) == g["count"]).all())
print("missing prices:", df["sell_price"].isna().sum())
print("event_name_1 values:", df["event_name_1"].nunique())
print("memory (MB):", round(df.memory_usage(deep=True).sum() / 1e6, 1))

rows: 4788267
every item consecutive: True
missing prices: 0
event_name_1 values: 31
memory (MB): 187.4


In [4]:
# Cell 3: Lag features, with a leakage check
LAGS = [28, 35, 42, 49, 56]
gs = df.groupby("item_id", observed=True)["sales"]
for L in LAGS:
    df[f"lag_{L}"] = gs.shift(L).astype("float32")

# leakage check
bad = 0
for it in df["item_id"].drop_duplicates().sample(20, random_state=1):
    s = df[df["item_id"] == it]
    a = s["lag_28"].to_numpy()[28:]
    b = s["sales"].to_numpy()[:-28].astype("float32")
    bad += int(not np.array_equal(a, b))
print("items where lag_28 differs from real sales 28 days earlier:", bad)

counts = df.groupby("item_id", observed=True).size()
print("lag_28 NaN rows:", int(df["lag_28"].isna().sum()),
      "| expected:", int(np.minimum(28, counts).sum()))

display(df[["item_id", "d_num", "sales", "lag_28", "lag_35"]].iloc[30:36])

items where lag_28 differs from real sales 28 days earlier: 0
lag_28 NaN rows: 85372 | expected: 85372


,item_id,d_num,sales,lag_28,lag_35
30,FOODS_1_001,31,0,0.0,NaN
31,FOODS_1_001,32,2,1.0,NaN
32,FOODS_1_001,33,1,4.0,NaN
33,FOODS_1_001,34,7,2.0,NaN
34,FOODS_1_001,35,1,0.0,NaN
35,FOODS_1_001,36,2,2.0,3.0


In [5]:
# Cell 4: Rolling statistics (shifted by 28)
df["_s28"] = df.groupby("item_id", observed=True)["sales"].shift(28)
df["_z28"] = (df["_s28"] == 0).astype("float32").where(df["_s28"].notna())

g28 = df.groupby("item_id", observed=True)["_s28"]
gz = df.groupby("item_id", observed=True)["_z28"]

def roll(grp, w, fn):
    r = getattr(grp.rolling(w, min_periods = w), fn)()
    return r.reset_index(level=0, drop=True).astype("float32")

for w in (7, 14, 28, 56):
    df[f"roll_mean_{w}"] = roll(g28, w, "mean")
for w in (7, 28):
    df[f"roll_std_{w}"] = roll(g28, w, "std")
df["zero_rate_28"] = roll(gz, 28, "mean")

df = df.drop(columns=["_s28", "_z28"])

# leakage / correctness check
bad = 0
for it in df["item_id"].drop_duplicates().sample(20, random_state=2):
    s = df[df["item_id"] == it]
    n = len(s)
    y = s["sales"].to_numpy().astype("float64")
    for w in (7, 28, 56):
        got = s[f"roll_mean_{w}"].to_numpy()[-1]
        if n >= 28 + w:
            bad += int(not np.isclose(got, y[n -28 - w: n -28].mean(), atol=1e-4))
        else:
            bad += int(not np.isnan(got))
    got = s["roll_std_28"].to_numpy()[-1]
    if n >= 56:
        bad += int(not np.isclose(got, y[n - 56: n -28].std(ddof = 1), atol=1e-4))
print("wrong rolling values in the check:", bad)

counts = df.groupby("item_id", observed=True).size()
for w in (7, 28, 56):
    print(f"roll_mean_{w} NaN:", int(df[f"roll_mean_{w}"].isna().sum()),
          "| expected:", int(np.minimum(28 + w -1, counts).sum()))

print("memory (MB):", round(df.memory_usage(deep=True).sum() / 1e6, 1))
display(df[["roll_mean_7", "roll_mean_28", "roll_std_28", "zero_rate_28"]].describe().round(3))

wrong rolling values in the check: 0
roll_mean_7 NaN: 103666 | expected: 103666
roll_mean_28 NaN: 167695 | expected: 167695
roll_mean_56 NaN: 253067 | expected: 253067
memory (MB): 417.2


,roll_mean_7,roll_mean_28,roll_std_28,zero_rate_28
count,4684601.000,4620572.000,4620572.000,4620572.000
mean,1.637,1.637,1.454,0.553
std,3.825,3.664,2.150,0.307
min,0.000,0.000,0.000,0.000
25%,0.143,0.250,0.518,0.286
50%,0.714,0.714,0.957,0.571
75%,1.714,1.607,1.624,0.821
max,304.857,225.071,131.728,1.000


In [6]:
# Cell 5: Calendar, event, SNAP and Christmas features
cal = (df.drop_duplicates("d_num")[["d_num", "date"]]
       .sort_values("d_num").set_index("d_num"))
cal["dow"] = cal["date"].dt.dayofweek.astype("int8")
cal["dom"] = cal["date"].dt.day.astype("int8")
cal["woy"] = cal["date"].dt.isocalendar().week.astype("int8")
cal["is_weekend"] = (cal["dow"] >= 5).astype("int8")
cal["is_christmas"] = ((cal["date"].dt.month == 12) & (cal["date"].dt.day == 25)).astype("int8")

n_before = len(df)
for c in ["dow", "dom", "woy", "is_weekend", "is_christmas"]:
    df[c] = df["d_num"].map(cal[c]).astype("int8")
assert len(df) == n_before

df["has_event"] = ((df["event_name_1"] != "none") |
                   (df["event_name_2"] != "none")).astype("int8")
df = df.drop(columns=["wday"])

# checks against the EDA
print("average sales by weekday (0 = Monday):")
print(df[df["is_christmas"] == 0].groupby("dow")["sales"].mean().round(3).to_string())

xmas = df[df["is_christmas"] == 1]
print("\nChristmas dates:", sorted(xmas["date"].dt.date.unique()))
print("Christmas units sold:", int(xmas["sales"].sum()))

print("\nday of month by SNAP flag:")
print(df.groupby("snap")["dom"].agg(["min", "max"]).to_string())

print("\nshare of rows on an event day:", round(df["has_event"].mean(), 3))
print("memory (MB):", round(df.memory_usage(deep=True).sum() / 1e6, 1))

average sales by weekday (0 = Monday):
dow
0    1.567
1    1.391
2    1.355
3    1.372
4    1.642
5    2.057
6    2.090

Christmas dates: [datetime.date(2011, 12, 25), datetime.date(2012, 12, 25), datetime.date(2013, 12, 25), datetime.date(2014, 12, 25), datetime.date(2015, 12, 25)]
Christmas units sold: 0

day of month by SNAP flag:
      min  max
snap          
0      11   31
1       1   10

share of rows on an event day: 0.081
memory (MB): 441.1


In [8]:
# Cell 6: Price features                                
assert df["sell_price"].notna().all() and (df["sell_price"] > 0).all(), "bad prices"
gp = df.groupby("item_id", observed=True)["sell_price"]

df["price_diff_7"] = (df["sell_price"] / gp.shift(7) -1).astype("float32")

p_prev = gp.shift(1)
prev_mean28 = (p_prev.groupby(df["item_id"], observed=True)
               .rolling(28, min_periods=28).mean()
               .reset_index(level=0, drop=True))
df["price_ratio_28"] = (df["sell_price"] / prev_mean28).astype("float32")

df["price_ratio_max"] = (df["sell_price"] / gp.cummax()).astype("float32")

# correctness check on 20 random items (last row of each)
bad = 0
for it in df["item_id"].drop_duplicates().sample(20, random_state=3):
    s = df[df["item_id"] == it]
    n = len(s)
    p = s["sell_price"].to_numpy().astype("float64")

    a = s["price_diff_7"].to_numpy()[-1]
    bad += int(not(np.isclose(a, p[-1] / p[-8] -1, atol=1e-4) if n > 7 else np.isnan(a)))

    b = s["price_ratio_28"].to_numpy()[-1]
    bad += int(not(np.isclose(b, p[-1] / p[-29:-1].mean(), atol=1e-4) if n > 28 else np.isnan(b)))

    c = s["price_ratio_max"].to_numpy()[-1]
    bad += int(not np.isclose(c, p[-1] / p.max(), atol=1e-4))

print("wrong price features in the check:", bad)

counts = df.groupby("item_id", observed=True).size()
print("price_diff_7 NaN :", int(df["price_diff_7"].isna().sum()),
      "| expected:", int(np.minimum(7, counts).sum()))
print("price_ratio_28 NaN :", int(df["price_ratio_28"].isna().sum()),
      "| expected:", int(np.minimum(28, counts).sum()))
print("price_ratio_max NaN:", int(df["price_ratio_max"].isna().sum()), "| expected: 0")

moved = df["price_diff_7"].dropna().abs() > 1e-6
print("\nshare of rows where the price differs from 7 days ago:", round(moved.mean(), 3))
display(df[["price_diff_7", "price_ratio_28", "price_ratio_max"]].describe().round(3))

wrong price features in the check: 0
price_diff_7 NaN : 21343 | expected: 21343
price_ratio_28 NaN : 85372 | expected: 85372
price_ratio_max NaN: 0 | expected: 0

share of rows where the price differs from 7 days ago: 0.009


,price_diff_7,price_ratio_28,price_ratio_max
count,4766924.000,4702895.000,4788267.000
mean,0.001,1.000,0.979
std,0.307,0.017,0.051
min,-0.996,0.004,0.004
25%,0.000,1.000,0.997
50%,0.000,1.000,1.000
75%,0.000,1.000,1.000
max,246.000,5.880,1.000


In [9]:
# Cell 7: Check the extremes and add a long-window price feature
# extreme price values
ext = (df["price_ratio_max"] < 0.1) | (df["price_diff_7"].abs() > 2)
print("rows with extreme price ratios:", int(ext.sum()), "| items:", df.loc[ext, "item_id"].nunique())
print("rows with price below 0.10    :", int((df["sell_price"] < 0.1).sum()))
display(df.loc[ext, ["item_id", "d_num", "sell_price", "price_diff_7", "price_ratio_max"]]
        .drop_duplicates("item_id").head(8))

# price level vs the previous ~6 months (past prices only)
p_prev = df.groupby("item_id", observed=True)["sell_price"].shift(1)
prev_mean182 = (p_prev.groupby(df["item_id"], observed=True)
                .rolling(182, min_periods=28).mean()
                .reset_index(level=0, drop=True))
df["price_ratio_182"] = (df["sell_price"] / prev_mean182).astype("float32")

counts = df.groupby("item_id", observed=True).size()
print("\nprice_ratio_182 NaN:", int(df["price_ratio_182"].isna().sum()),
      "| expected:", int(np.minimum(28, counts).sum()))

r = df["price_ratio_182"].dropna()
print("rows more than 3% away from the 182-day mean:", round(((r -1).abs() > 0.03).mean(), 3))
display(r.describe().round(3).to_frame())

rows with extreme price ratios: 175 | items: 22
rows with price below 0.10    : 14


,item_id,d_num,sell_price,price_diff_7,price_ratio_max
259296,FOODS_1_158,1800,4.680000,2.492537,0.939759
631678,FOODS_2_177,1198,3.330000,2.330000,1.000000
2778147,HOBBIES_1_333,953,4.970000,4.123711,1.000000
3017846,HOBBIES_2_062,442,5.880000,3.000000,1.000000
3036912,HOBBIES_2_075,1513,0.050000,-0.982639,0.017361
3266628,HOUSEHOLD_1_077,1450,6.420000,5.420000,0.919771
3315606,HOUSEHOLD_1_109,1366,3.970000,2.970000,1.000000
3399695,HOUSEHOLD_1_165,1646,17.969999,7.985000,1.000000



price_ratio_182 NaN: 85372 | expected: 85372
rows more than 3% away from the 182-day mean: 0.084


,price_ratio_182
count,4702895.000
mean,1.002
std,0.028
min,0.004
25%,1.000
50%,1.000
75%,1.000
max,2.596


In [10]:
# Cell 8: Leakage test, feature list and NaN report
def sales_feats(sales):
    x = pd.Series(np.asarray(sales, dtype="float64"))
    sh = x.shift(28)
    return pd.DataFrame({
        "lag_28": sh,
        "roll_mean_28": sh.rolling(28, min_periods=28).mean(),
        "roll_std_28": sh.rolling(28, min_periods=28).std(),
        "zero_rate_28": (sh == 0).astype(float).where(sh.notna()).rolling(28, min_periods=28).mean(),
    })

counts = df.groupby("item_id", observed=True).size()
test_items = pd.Series(counts[counts >= 400].index).sample(10, random_state=4)

mismatch = leaks = undetected = 0
for it in test_items:
    s = df[df["item_id"] == it]
    y = s["sales"].to_numpy()
    n = len(y)
    D = n // 2

    base = sales_feats(y)
    stored = s[["lag_28", "roll_mean_28", "roll_std_28", "zero_rate_28"]].to_numpy().astype("float64")
    mismatch += int(not np.allclose(base.to_numpy(), stored, atol=1e-3, equal_nan=True))

    y2 = y.copy()
    y2[D:] = 999
    new = sales_feats(y2)
    leaks += int(not np.allclose(base.iloc[:D + 28].to_numpy(), new.iloc[:D + 28].to_numpy(), equal_nan=True))
    undetected += int(np.allclose(base.iloc[D + 28:].to_numpy(), new.iloc[D + 28:].to_numpy(), equal_nan=True))

print("items tested                           :", len(test_items))
print("function differs from stored columns   :", mismatch, "(expect 0)")
print("features changed before day D+28 (leak):", leaks, "(expect 0)")
print("items where the test saw no change     :", undetected, "(expect 0)")

# feature list
NUM = ["sell_price", "price_diff_7", "price_ratio_28", "price_ratio_max", "price_ratio_182",
       "lag_28", "lag_35", "lag_42", "lag_49", "lag_56",
       "roll_mean_7", "roll_mean_14", "roll_mean_28", "roll_mean_56",
       "roll_std_7", "roll_std_28", "zero_rate_28",
       "age", "dow", "dom", "woy", "month", "is_weekend", "is_christmas", "has_event", "snap"]
CAT = ["item_id", "dept_id", "cat_id", "event_name_1", "event_type_1", "event_name_2", "event_type_2"]
FEATURES = NUM + CAT
TARGET = "sales"

forbidden = {"sales", "d_num", "date", "first_day", "year", "id", "store_id", "state_id"}
assert not (set(FEATURES) & forbidden), "forbidden column in the feature list"
assert all(c in df.columns for c in FEATURES + [TARGET]), "missing column"
print("\nfeatures:", len(NUM), "numeric +", len(CAT), "categorical")

# NaN report
rep = pd.DataFrame({"dtype": df[FEATURES].dtypes.astype(str),
                    "NaN": df[FEATURES].isna().sum(),
                    "NaN_%": (df[FEATURES].isna().mean() * 100).round(2)})
display(rep[rep["NaN"] > 0])
print("features with no NaN:", int((rep["NaN"] == 0).sum()), "of", len(FEATURES))
print("memory (MB):", round(df.memory_usage(deep=True).sum() / 1e6, 1))

items tested                           : 10
function differs from stored columns   : 0 (expect 0)
features changed before day D+28 (leak): 0 (expect 0)
items where the test saw no change     : 0 (expect 0)

features: 26 numeric + 7 categorical


,dtype,NaN,NaN_%
price_diff_7,float32,21343,0.45
price_ratio_28,float32,85372,1.78
price_ratio_182,float32,85372,1.78
lag_28,float32,85372,1.78
lag_35,float32,106715,2.23
lag_42,float32,128058,2.67
lag_49,float32,149401,3.12
lag_56,float32,170744,3.57
roll_mean_7,float32,103666,2.17
roll_mean_14,float32,125009,2.61


features with no NaN: 18 of 33
memory (MB): 517.7


In [13]:
# Cell 9: Save the feature table and reload it with a value check
import json
from pathlib import Path

KEEP = ["d_num", "date", TARGET] + FEATURES
out_path = Path("../data/processed/features.parquet")
meta_path = Path("../data/processed/features_meta.json")

df[KEEP].to_parquet(out_path, index=False)

meta = {"NUM": NUM, "CAT": CAT, "TARGET": TARGET, "H": H, "LAST_DAY": LAST_DAY}
meta_path.write_text(json.dumps(meta, indent=2))

chk = pd.read_parquet(out_path)
print("saved:", out_path.resolve())
print("file size (MB):", round(out_path.stat().st_size / 1e6, 1))
print("shape:", chk.shape, "| same as df[KEEP]:", chk.shape == df[KEEP].shape)
print("dtypes preserved:", (chk.dtypes == df[KEEP].dtypes).all())
print("total sales equal:", int(chk["sales"].sum()) == int(df["sales"].sum()))
print("order equal (d_num):", bool((chk["d_num"].to_numpy() == df["d_num"].to_numpy()).all()))
print("NaN counts equal:", bool((chk[NUM].isna().sum() == df[NUM].isna().sum()).all()))
print("numeric values identical:", all(
    np.allclose(chk[c].to_numpy("float64"), df[c].to_numpy("float64"), equal_nan=True)
    for c in NUM))
print("categorical values identical:", all(
    chk[c].astype(str).equals(df[c].astype(str)) for c in CAT))
del chk

saved: C:\Users\ASUS\Desktop\AI_Projects\Retail-demand-forecasting\data\processed\features.parquet
file size (MB): 69.3
shape: (4788267, 36) | same as df[KEEP]: True
dtypes preserved: True
total sales equal: True
order equal (d_num): True
NaN counts equal: True
numeric values identical: True
categorical values identical: True


## Summary of Notebook 05: feature engineering

**Design rule:** every sales-based feature uses only sales up to day t-28, so one model predicts all 28 days of the horizon from the same cutoff (no recursion).

| Group | Features |
|---|---|
| Lags | lag_28, 35, 42, 49, 56 (multiples of 7: same weekday as the target day) |
| Rolling (shifted by 28) | mean over 7, 14, 28, 56 days; std over 7, 28; zero_rate_28 |
| Calendar | dow, dom (paycheck/SNAP timing), woy, month, is_weekend, age (days since launch) |
| Events | event_name_1/2, event_type_1/2 (label "none"), has_event, snap, is_christmas |
| Price | sell_price, price_diff_7, price_ratio_28, price_ratio_182, price_ratio_max |
| Categorical | item_id, dept_id, cat_id |

26 numeric + 7 categorical features, 4,788,267 rows (all 3,049 items, launched days only).

**Checks passed**
- Each item is consecutive, so shift(k) means exactly k days earlier
- Lag and rolling values match hand calculations; NaN counts match the expected number
- Perturbation test (future sales replaced by 999): no feature changed before day D+28 (tested on lag_28, roll_mean_28, roll_std_28, zero_rate_28)
- Calendar checks: weekday pattern matches the EDA, Christmas has 0 units, SNAP = days 1-10 of the month

**Bugs found and fixed:** `dom` was built from the weekday number instead of the day of the month (caught by the SNAP check)

**NaN rules:** pre-launch rows dropped; lags/rolls/price changes at the start of an item's history stay NaN (LightGBM handles them); no-event days labelled "none"

**Findings**
- Prices change rarely: 0.9% of rows differ from 7 days ago, but 8.4% are more than 3% from the 182-day mean, so the long-window price ratio carries information the short ones miss
- 175 rows (22 items) have extreme price jumps; kept

**Limits (for the README)**
- Future prices are assumed known at forecast time (planned prices)
- 25 December is a store closure (0 units); the zero stays in later lag/rolling windows
- `year` is not used as a feature (trend would not extrapolate to the test period)
- Perturbation test covers 4 of the 17 sales-based features; the rest share the same shift code
- item_id has 3,049 levels; to be tested with and without in Notebook 06